In [4]:
import pandas as pd
import geopandas as gpd

FPATH = "/Users/maxbdixon/Downloads/npi_raw_201912/npidata_pfile_20050523-20191219.csv"

COLS = [
    "NPI",
    "Entity Type Code",
    "Provider Credential Text",
    "Provider Business Practice Location Address State Name",
    "Provider First Line Business Practice Location Address",
    "Provider Business Practice Location Address City Name",
    "Provider Business Practice Location Address Postal Code",
    "Healthcare Provider Taxonomy Code_1",
    "Healthcare Provider Primary Taxonomy Switch_1",
]

# ── 1. Read in chunks, filtering to TN on the fly ────────────────────────────
chunks = []
for chunk in pd.read_csv(FPATH, usecols=COLS, dtype=str, chunksize=100_000):
    chunk = chunk[
        (chunk["Entity Type Code"] == "1") &
        (chunk["Provider Business Practice Location Address State Name"] == "TN")
    ]
    chunks.append(chunk)

df = pd.concat(chunks, ignore_index=True)
print(f"TN individual providers: {len(df)}")

# ── 2. Filter to physicians via NUCC taxonomy codes ───────────────────────────
is_primary = df["Healthcare Provider Primary Taxonomy Switch_1"] == "Y"
is_physician = df["Healthcare Provider Taxonomy Code_1"].str.startswith("207", na=False)

df_phys = df[is_primary & is_physician].copy()
print(f"TN physicians: {len(df_phys)}")
print(df_phys["Healthcare Provider Taxonomy Code_1"].value_counts().head(10))

# ── 3. Geocode using Census Geocoder (batch) ──────────────────────────────────
import requests
import io

def census_batch_geocode(df_batch, batch_id):
    """
    Submit a batch of up to 10,000 addresses to the Census geocoder.
    Input df must have columns: id, street, city, state, zip
    """
    url = "https://geocoding.geo.census.gov/geocoder/locations/addressbatch"
    
    # Format as required CSV: ID, Street, City, State, ZIP
    csv_data = df_batch[["id", "street", "city", "state", "zip"]].to_csv(
        index=False, header=False
    )
    
    response = requests.post(
        url,
        files={"addressFile": ("addresses.csv", csv_data, "text/csv")},
        data={
            "benchmark": "Public_AR_Census2020",
            "vintage": "Census2020_Census2020",
        },
        timeout=300,
    )
    
    if response.status_code != 200:
        print(f"Batch {batch_id} failed: {response.status_code}")
        return pd.DataFrame()
    
    result = pd.read_csv(
        io.StringIO(response.text),
        header=None,
        names=["id", "input_address", "match", "match_type", "matched_address",
               "coords", "tiger_line_id", "tiger_side"],
        dtype=str,
    )
    return result

# Prep address dataframe
df_phys = df_phys.reset_index(drop=True)
df_addr = pd.DataFrame({
    "id": df_phys.index.astype(str),
    "street": df_phys["Provider First Line Business Practice Location Address"].fillna(""),
    "city": df_phys["Provider Business Practice Location Address City Name"].fillna(""),
    "state": df_phys["Provider Business Practice Location Address State Name"].fillna(""),
    "zip": df_phys["Provider Business Practice Location Address Postal Code"].str[:5].fillna(""),
})

# Run in batches of 9,500 (stay under the 10k limit)
BATCH_SIZE = 9_500
results = []
n_batches = (len(df_addr) // BATCH_SIZE) + 1

for i in range(n_batches):
    batch = df_addr.iloc[i * BATCH_SIZE : (i + 1) * BATCH_SIZE]
    if len(batch) == 0:
        continue
    print(f"Geocoding batch {i+1}/{n_batches} ({len(batch)} records)...")
    result = census_batch_geocode(batch, i)
    results.append(result)

geo_results = pd.concat(results, ignore_index=True)

# ── 4. Parse coordinates from results ────────────────────────────────────────
matched = geo_results[geo_results["match"] == "Match"].copy()
matched[["lon", "lat"]] = matched["coords"].str.split(",", expand=True).astype(float)
matched["id"] = matched["id"].astype(int)

print(f"Successfully geocoded: {len(matched)} / {len(df_phys)}")

# ── 5. Join back to physician dataframe ──────────────────────────────────────
df_phys = df_phys.reset_index(drop=True)
df_phys.index.name = "id"
df_phys = df_phys.merge(
    matched[["id", "lat", "lon"]],
    left_index=True,
    right_on="id",
    how="left",
)
df_phys = df_phys.dropna(subset=["lat", "lon"])

# ── 6. Spatial join to tracts ─────────────────────────────────────────────────
gdf_phys = gpd.GeoDataFrame(
    df_phys,
    geometry=gpd.points_from_xy(df_phys["lon"], df_phys["lat"]),
    crs="EPSG:4326",
)

# Load your tract GeoDataFrame
gdf_tracts = gpd.read_file("/Users/maxbdixon/Documents/GitHub/SoVI-clustering/V2/results/K-SoVI_UMAP_2019_Tennessee.gpkg")
gdf_tracts = gdf_tracts.to_crs("EPSG:4326")

joined = gpd.sjoin(
    gdf_phys[["NPI", "geometry"]],
    gdf_tracts[["GEOID", "geometry"]],
    how="left",
    predicate="within",
)

physician_counts = (
    joined.groupby("GEOID")["NPI"]
    .count()
    .reset_index()
    .rename(columns={"NPI": "physician_count"})
)

# ── 7. Merge onto tracts and compute rate ─────────────────────────────────────
gdf_tracts = gdf_tracts.merge(physician_counts, on="GEOID", how="left")
gdf_tracts["physician_count"] = gdf_tracts["physician_count"].fillna(0)

# Assumes your ACS data has a total_pop column already on gdf_tracts
gdf_tracts["physicians_per_10k"] = (
    gdf_tracts["physician_count"] / gdf_tracts["TOTALPOP"] * 10_000
).where(gdf_tracts["TOTALPOP"] > 0)

print(gdf_tracts[["GEOID", "physician_count", "physicians_per_10k"]].describe())

TN individual providers: 85248
TN physicians: 10343
Healthcare Provider Taxonomy Code_1
207R00000X    2116
207Q00000X    2033
207P00000X     893
207L00000X     710
207V00000X     653
207X00000X     427
207RC0000X     343
207ZP0102X     321
207W00000X     308
207RG0100X     288
Name: count, dtype: int64
Geocoding batch 1/2 (9500 records)...
Geocoding batch 2/2 (843 records)...
Successfully geocoded: 8461 / 10343
       physician_count  physicians_per_10k
count      1471.000000         1471.000000
mean          5.711081           15.933202
std          22.615091           99.355803
min           0.000000            0.000000
25%           0.000000            0.000000
50%           0.000000            0.000000
75%           3.000000            6.025374
max         429.000000         3023.872679


In [6]:
# which tract is the outlier?
print(gdf_tracts.nlargest(5, "physicians_per_10k")[["GEOID", "physician_count", "physicians_per_10k", "TOTALPOP"]])

           GEOID  physician_count  physicians_per_10k  TOTALPOP
153  47157003800            228.0         3023.872679       754
942  47157003600            147.0          955.786736      1538
352  47037016500            429.0          886.180541      4841
774  47037014400            186.0          881.099005      2111
697  47065001200            216.0          706.344016      3058


In [8]:
p99 = gdf_tracts["physicians_per_10k"].quantile(0.99)
gdf_tracts["physicians_per_10k_w"] = gdf_tracts["physicians_per_10k"].clip(upper=p99)
print(f"99th percentile cap: {p99:.1f}")

99th percentile cap: 309.0


In [9]:
for p in [90, 95, 97, 99]:
    print(f"{p}th percentile: {gdf_tracts['physicians_per_10k'].quantile(p/100):.1f}")

90th percentile: 24.4
95th percentile: 51.4
97th percentile: 89.3
99th percentile: 309.0


In [12]:
import re

# get the unmatched IDs from geo_results
unmatched_ids = geo_results[geo_results["match"] != "Match"]["id"].astype(int)

# rebuild address df from original df_phys before the merge overwrote it
# you'll need to re-run the filter steps to get df_phys back, then:
df_retry = df_addr.loc[df_addr["id"].astype(int).isin(unmatched_ids)].copy()

# strip suite/unit numbers
df_retry["street"] = df_retry["street"].str.replace(
    r"(?i)\s+(ste|suite|unit|apt|#|floor|fl|rm|room)\.?\s+\S+", "", regex=True
).str.strip()

# also trim zip to 5 digits if not already
df_retry["zip"] = df_retry["zip"].str[:5]

print(f"Retrying {len(df_retry)} unmatched addresses...")

# resubmit in batches
retry_results = []
for i in range(0, len(df_retry), 9500):
    batch = df_retry.iloc[i:i+9500]
    print(f"Retry batch {i//9500 + 1}...")
    result = census_batch_geocode(batch, f"retry_{i}")
    retry_results.append(result)

geo_retry = pd.concat(retry_results, ignore_index=True)
print(geo_retry["match"].value_counts())

Retrying 1882 unmatched addresses...
Retry batch 1...
match
No_Match    1706
Tie          176
Name: count, dtype: int64


In [13]:
print(geo_results[geo_results["match"] != "Match"]["input_address"].str.extract(r',\s*([A-Z]+)\s*,')[0].value_counts().head(10))

0
NASHVILLE       896
MEMPHIS         189
TN              124
JACKSON          90
HERMITAGE        73
BRISTOL          59
MURFREESBORO     36
KNOXVILLE        31
CHATTANOOGA      25
SPRINGFIELD      25
Name: count, dtype: int64


In [14]:
physician_export = gdf_tracts[["GEOID", "physician_count", "physicians_per_10k", "physicians_per_10k_w", "geometry"]].copy()

physician_export.to_file("/Users/maxbdixon/Desktop/SCHOOL/BRIC data/tn_physicians_per_10k.gpkg", driver="GPKG")
print(f"Exported {len(physician_export)} tracts")
print(physician_export.drop(columns="geometry").describe())

Exported 1471 tracts
       physician_count  physicians_per_10k  physicians_per_10k_w
count      1471.000000         1471.000000           1471.000000
mean          5.711081           15.933202             12.008954
std          22.615091           99.355803             39.999450
min           0.000000            0.000000              0.000000
25%           0.000000            0.000000              0.000000
50%           0.000000            0.000000              0.000000
75%           3.000000            6.025374              6.025374
max         429.000000         3023.872679            309.008540


In [15]:
import geopandas as gpd

# ── 1. Load NCES school locations ─────────────────────────────────────────────
gdf_schools = gpd.read_file(
    "/Users/maxbdixon/Downloads/EDGE_GEOCODE_PUBLICSCH_1819/Shapefile_SCH/EDGE_GEOCODE_PUBLICSCH_1819.shp"
)

print(gdf_schools.shape)
print(gdf_schools.columns.tolist())
print(gdf_schools.head(2))

(102176, 26)
['NCESSCH', 'LEAID', 'NAME', 'OPSTFIPS', 'STREET', 'CITY', 'STATE', 'ZIP', 'STFIP', 'CNTY', 'NMCNTY', 'LOCALE', 'LAT', 'LON', 'CBSA', 'NMCBSA', 'CBSATYPE', 'CSA', 'NMCSA', 'NECTA', 'NMNECTA', 'CD', 'SLDL', 'SLDU', 'SCHOOLYEAR', 'geometry']
        NCESSCH    LEAID                       NAME OPSTFIPS  \
0  010000500870  0100005  Albertville Middle School       01   
1  010000500871  0100005    Albertville High School       01   

              STREET         CITY STATE    ZIP STFIP   CNTY  ... CBSATYPE  \
0  600 E Alabama Ave  Albertville    AL  35950    01  01095  ...        2   
1   402 E McCord Ave  Albertville    AL  35950    01  01095  ...        2   

   CSA                               NMCSA  NECTA NMNECTA    CD   SLDL   SLDU  \
0  290  Huntsville-Decatur-Albertville, AL      N       N  0104  01026  01009   
1  290  Huntsville-Decatur-Albertville, AL      N       N  0104  01026  01009   

  SCHOOLYEAR                    geometry  
0  2018-2019  POINT (-86.20617 34.2

In [16]:
import geopandas as gpd

# ── 1. Filter to TN ───────────────────────────────────────────────────────────
gdf_schools_tn = gdf_schools[gdf_schools["STFIP"] == "47"].copy()
print(f"TN schools: {len(gdf_schools_tn)}")

# ── 2. Spatial join to tracts ─────────────────────────────────────────────────
gdf_schools_tn = gdf_schools_tn.to_crs(gdf_tracts.crs)

joined = gpd.sjoin(
    gdf_schools_tn[["NCESSCH", "geometry"]],
    gdf_tracts[["GEOID", "geometry"]],
    how="left",
    predicate="within",
)

school_counts = (
    joined.groupby("GEOID")["NCESSCH"]
    .count()
    .reset_index()
    .rename(columns={"NCESSCH": "school_count"})
)

# ── 3. Merge onto tracts and compute rate ─────────────────────────────────────
gdf_tracts = gdf_tracts.merge(school_counts, on="GEOID", how="left")
gdf_tracts["school_count"] = gdf_tracts["school_count"].fillna(0)

gdf_tracts["schools_per_10k"] = (
    gdf_tracts["school_count"] / gdf_tracts["TOTALPOP"] * 10_000
).where(gdf_tracts["TOTALPOP"] > 0)

print(gdf_tracts[["GEOID", "school_count", "schools_per_10k"]].describe())

# ── 4. Export ─────────────────────────────────────────────────────────────────
school_export = gdf_tracts[["GEOID", "school_count", "schools_per_10k", "geometry"]].copy()
school_export.to_file("/Users/maxbdixon/Desktop/SCHOOL/BRIC data/tn_schools_per_10k.gpkg", driver="GPKG")
print("Exported tn_schools_per_10k.gpkg")

TN schools: 1874
       school_count  schools_per_10k
count   1471.000000      1471.000000
mean       1.271244         3.000930
std        1.259190         3.220685
min        0.000000         0.000000
25%        0.000000         0.000000
50%        1.000000         2.403846
75%        2.000000         4.520819
max        8.000000        31.746032
Exported tn_schools_per_10k.gpkg


In [22]:
import pandas as pd

df_bmf = pd.read_csv(
    "/Users/maxbdixon/Downloads/UNIFIED_BMF_V1.2.csv",
    dtype=str,
    low_memory=False
)

print(df_bmf.shape)
print(df_bmf.columns.tolist())

(3516801, 50)
['EIN2', 'EIN', 'NTEE_IRS', 'NTEE_NCCS', 'NTEEV2', 'NCCS_LEVEL_1', 'NCCS_LEVEL_2', 'NCCS_LEVEL_3', 'F990_ORG_ADDR_CITY', 'F990_ORG_ADDR_STATE', 'F990_ORG_ADDR_ZIP', 'F990_ORG_ADDR_STREET', 'CENSUS_CBSA_FIPS', 'CENSUS_CBSA_NAME', 'CENSUS_BLOCK_FIPS', 'CENSUS_URBAN_AREA', 'CENSUS_STATE_ABBR', 'CENSUS_COUNTY_NAME', 'ORG_ADDR_FULL', 'ORG_ADDR_MATCH', 'LATITUDE', 'LONGITUDE', 'GEOCODER_SCORE', 'GEOCODER_MATCH', 'BMF_SUBSECTION_CODE', 'BMF_STATUS_CODE', 'BMF_PF_FILING_REQ_CODE', 'BMF_ORGANIZATION_CODE', 'BMF_INCOME_CODE', 'BMF_GROUP_EXEMPT_NUM', 'BMF_FOUNDATION_CODE', 'BMF_FILING_REQ_CODE', 'BMF_DEDUCTIBILITY_CODE', 'BMF_CLASSIFICATION_CODE', 'BMF_ASSET_CODE', 'BMF_AFFILIATION_CODE', 'ORG_RULING_DATE', 'ORG_FISCAL_YEAR', 'ORG_RULING_YEAR', 'ORG_YEAR_FIRST', 'ORG_YEAR_LAST', 'ORG_YEAR_COUNT', 'ORG_PERS_ICO', 'ORG_NAME_SEC', 'ORG_NAME_CURRENT', 'ORG_FISCAL_PERIOD', 'F990_TOTAL_REVENUE_RECENT', 'F990_TOTAL_INCOME_RECENT', 'F990_TOTAL_ASSETS_RECENT', 'F990_TOTAL_EXPENSES_RECENT']


In [23]:
import pandas as pd
import geopandas as gpd

# ── 1. Filter to TN orgs active in 2019 ──────────────────────────────────────
df_bmf_tn = df_bmf[
    (df_bmf["CENSUS_STATE_ABBR"] == "TN") &
    (df_bmf["ORG_YEAR_FIRST"].astype(float) <= 2019) &
    (df_bmf["ORG_YEAR_LAST"].astype(float) >= 2019)
].copy()

print(f"TN orgs active in 2019: {len(df_bmf_tn)}")

# ── 2. Filter to civic (S) and religious (X) NTEE codes ───────────────────────
# NTEE_NCCS is cleaner than NTEE_IRS
civic_religious = df_bmf_tn[
    df_bmf_tn["NTEE_NCCS"].str.startswith("S", na=False) |
    df_bmf_tn["NTEE_NCCS"].str.startswith("X", na=False)
].copy()

print(f"Civic (S) orgs: {df_bmf_tn['NTEE_NCCS'].str.startswith('S', na=False).sum()}")
print(f"Religious (X) orgs: {df_bmf_tn['NTEE_NCCS'].str.startswith('X', na=False).sum()}")
print(f"Total civic + religious: {len(civic_religious)}")

# ── 3. Drop missing coordinates ───────────────────────────────────────────────
civic_religious = civic_religious.dropna(subset=["LATITUDE", "LONGITUDE"])
civic_religious["LATITUDE"] = civic_religious["LATITUDE"].astype(float)
civic_religious["LONGITUDE"] = civic_religious["LONGITUDE"].astype(float)
civic_religious = civic_religious[
    (civic_religious["LATITUDE"] != 0) &
    (civic_religious["LONGITUDE"] != 0)
]
print(f"After dropping missing coords: {len(civic_religious)}")

# ── 4. Spatial join to tracts ─────────────────────────────────────────────────
gdf_orgs = gpd.GeoDataFrame(
    civic_religious,
    geometry=gpd.points_from_xy(civic_religious["LONGITUDE"], civic_religious["LATITUDE"]),
    crs="EPSG:4326",
)

gdf_tracts = gdf_tracts.to_crs("EPSG:4326")

joined = gpd.sjoin(
    gdf_orgs[["EIN", "NTEE_NCCS", "geometry"]],
    gdf_tracts[["GEOID", "geometry"]],
    how="left",
    predicate="within",
)

# ── 5. Counts by tract ────────────────────────────────────────────────────────
civic_counts = (
    joined[joined["NTEE_NCCS"].str.startswith("S", na=False)]
    .groupby("GEOID")["EIN"].count()
    .reset_index()
    .rename(columns={"EIN": "civic_org_count"})
)

religious_counts = (
    joined[joined["NTEE_NCCS"].str.startswith("X", na=False)]
    .groupby("GEOID")["EIN"].count()
    .reset_index()
    .rename(columns={"EIN": "religious_org_count"})
)

# ── 6. Merge onto tracts and compute rates ────────────────────────────────────
gdf_tracts = gdf_tracts.merge(civic_counts, on="GEOID", how="left")
gdf_tracts = gdf_tracts.merge(religious_counts, on="GEOID", how="left")
gdf_tracts["civic_org_count"] = gdf_tracts["civic_org_count"].fillna(0)
gdf_tracts["religious_org_count"] = gdf_tracts["religious_org_count"].fillna(0)

gdf_tracts["civic_orgs_per_10k"] = (
    gdf_tracts["civic_org_count"] / gdf_tracts["TOTALPOP"] * 10_000
).where(gdf_tracts["TOTALPOP"] > 0)

gdf_tracts["religious_orgs_per_10k"] = (
    gdf_tracts["religious_org_count"] / gdf_tracts["TOTALPOP"] * 10_000
).where(gdf_tracts["TOTALPOP"] > 0)

print(gdf_tracts[["civic_org_count", "civic_orgs_per_10k",
                   "religious_org_count", "religious_orgs_per_10k"]].describe())

# ── 7. Export ─────────────────────────────────────────────────────────────────
org_export = gdf_tracts[["GEOID", "civic_org_count", "civic_orgs_per_10k",
                          "religious_org_count", "religious_orgs_per_10k",
                          "geometry"]].copy()
org_export.to_file("/Users/maxbdixon/Desktop/SCHOOL/BRIC data/tn_orgs_per_10k.gpkg", driver="GPKG")
print("Exported tn_orgs_per_10k.gpkg")

TN orgs active in 2019: 35386
Civic (S) orgs: 2119
Religious (X) orgs: 5586
Total civic + religious: 7705
After dropping missing coords: 7705
       civic_org_count  civic_orgs_per_10k  religious_org_count  \
count      1471.000000         1471.000000          1471.000000   
mean          1.424881            3.579890             3.787899   
std           2.720225            7.894553             3.802060   
min           0.000000            0.000000             0.000000   
25%           0.000000            0.000000             1.000000   
50%           1.000000            1.663617             3.000000   
75%           2.000000            4.277161             5.000000   
max          57.000000          152.259332            41.000000   

       religious_orgs_per_10k  
count             1471.000000  
mean                 8.993421  
std                 10.050040  
min                  0.000000  
25%                  3.491631  
50%                  6.499415  
75%                 11.332517 

In [24]:
print(gdf_tracts.nlargest(5, "civic_orgs_per_10k")[["GEOID", "civic_org_count", "TOTALPOP", "civic_orgs_per_10k"]])

            GEOID  civic_org_count  TOTALPOP  civic_orgs_per_10k
1369  47065003100             31.0      2036          152.259332
193   47187050304             24.0      2464           97.402597
964   47093000100             22.0      2463           89.321965
793   47037019500             57.0      7956           71.644042
1181  47157004200             21.0      3323           63.195907


In [25]:
for col in ["civic_orgs_per_10k", "religious_orgs_per_10k"]:
    p97 = gdf_tracts[col].quantile(0.97)
    print(f"{col} 97th percentile: {p97:.1f}")
    gdf_tracts[f"{col}_w"] = gdf_tracts[col].clip(upper=p97)

civic_orgs_per_10k 97th percentile: 17.2
religious_orgs_per_10k 97th percentile: 29.5


In [26]:
org_export = gdf_tracts[["GEOID", "civic_org_count", "civic_orgs_per_10k", "civic_orgs_per_10k_w",
                          "religious_org_count", "religious_orgs_per_10k", "religious_orgs_per_10k_w",
                          "geometry"]].copy()
org_export.to_file("/Users/maxbdixon/Desktop/SCHOOL/BRIC data/tn_orgs_per_10k.gpkg", driver="GPKG")
print("Exported tn_orgs_per_10k.gpkg")

Exported tn_orgs_per_10k.gpkg


In [28]:
import pandas as pd

df_pos = pd.read_csv(
    "/Users/maxbdixon/Downloads/POS_File_Hospital_Non_Hospital_Facilities_Q4_2019.csv",
    dtype=str,
    low_memory=False,
    encoding="latin-1"
)
# ── 2. Filter to TN psychiatric/mental health facilities ──────────────────────
psych_codes = ["32", "33", "71", "76"]

df_mh = df_pos[
    (df_pos["STATE_CD"] == "TN") &
    (df_pos["PRVDR_CTGRY_CD"].isin(psych_codes))
].copy()

print(f"TN mental health facilities: {len(df_mh)}")
print(df_mh["PRVDR_CTGRY_CD"].value_counts())
print(df_mh[["FAC_NAME", "ST_ADR", "CITY_NAME", "ZIP_CD"]].head(10))

TN mental health facilities: 0
Series([], Name: count, dtype: int64)
Empty DataFrame
Columns: [FAC_NAME, ST_ADR, CITY_NAME, ZIP_CD]
Index: []


In [31]:
tn = df_pos[df_pos["STATE_CD"] == "TN"]
print(f"Total TN facilities: {len(tn)}")
print(tn["PRVDR_CTGRY_CD"].value_counts())

Total TN facilities: 3109
PRVDR_CTGRY_CD
05    455
02    317
12    316
10    310
09    274
01    270
08    213
15    197
11    193
21    183
16    103
04     63
03     56
14     46
07     43
19     38
06     29
17      3
Name: count, dtype: int64


In [32]:
# look at facility names for each code to identify psychiatric ones
for code in tn["PRVDR_CTGRY_CD"].unique():
    sample = tn[tn["PRVDR_CTGRY_CD"] == code]["FAC_NAME"].head(3).tolist()
    print(f"\nCode {code}: {sample}")


Code 01: ['UNICOI COUNTY  HOSPITAL', 'JACKSON-MADISON COUNTY GENERAL HOSPITAL', 'SUMNER REGIONAL MEDICAL CENTER']

Code 16: ['HOSPICE OF MURFREESBORO', 'ALIVE HOSPICE INC', 'METHODIST ALLIANCE HOSPICE']

Code 21: ['LIFESPAN HEALTH', 'LIMESTONE MEDICAL CENTER', 'LIMESTONE MEDICAL CENTER']

Code 09: ['RHEA COUNTY MEDICAL CENTER', '(CLOSED) VANDERBILT UNIVERSITY HOSP', 'REN CENTER MEMPHIS CENTRAL']

Code 12: ['(CLOSED) WATAUGA FAMILY HEALTH CLINIC', '(CLOSED) HAMPTON FAMILY HEALTH CLINIC', '(CLOSED) DEER LODGE MEDICAL CLINIC']

Code 14: ['FIRST TENNESSEE REHABILITATION CENTER', 'MOSE AND GARRISON SISKIN MEM FOUND', 'DANIEL ARTHUR REHABILITATION CENTER']

Code 19: ['(CLOSED) HELEN ROSS MCNABB CENTER INC', 'RIDGEVIEW PSYCHIATRIC HOSPITAL', '(CLOSED) OVERLOOK CENTER INC']

Code 02: ['NHC HEALTHCARE, OAKWOOD', 'NHC HEALTHCARE, DICKSON', 'ST BARNABAS AT SISKIN HOSPITAL']

Code 03: ['(CLOSED) MCKENDREE VILLAGE INC', '(CLOSED) IMPERIAL GARDENS HEALTH AND REHABILIATION', '(CLOSED) NHC HEALTHCARE

In [33]:
# psychiatric hospitals via subtype code
psych_subtypes = tn[
    (tn["PRVDR_CTGRY_CD"] == "01") &
    (tn["PRVDR_CTGRY_SBTYP_CD"].isin(["04", "07", "24", "25"]))
][["FAC_NAME", "PRVDR_CTGRY_SBTYP_CD", "ST_ADR", "CITY_NAME", "ZIP_CD"]]

print(f"Psychiatric hospitals (cat 01, subtypes 04/07/24/25): {len(psych_subtypes)}")
print(psych_subtypes.head(10))

# category 19 - likely community mental health centers
cat19 = tn[tn["PRVDR_CTGRY_CD"] == "19"][["FAC_NAME", "PRVDR_CTGRY_SBTYP_CD", "ST_ADR", "CITY_NAME", "ZIP_CD"]]
print(f"\nCategory 19 facilities: {len(cat19)}")
print(cat19)

Psychiatric hospitals (cat 01, subtypes 04/07/24/25): 29
                                         FAC_NAME PRVDR_CTGRY_SBTYP_CD  \
112474          LAKESHORE MENTAL HEALTH INSTITUTE                   04   
112475            MEMPHIS MENTAL HEALTH INSTITUTE                   04   
112476     MOCCASSIN BEND MENTAL HEALTH INSTITUTE                   04   
112477  RIDGEVIEW PSYCHIATRIC HOSPITAL AND CENTER                   04   
112478          LAKESIDE BEHAVIORAL HEALTH SYSTEM                   04   
112479     BEHAVIORAL HEALTHCARE CENTER AT MARTIN                   04   
112480                (CLOSED) WOODRIDGE HOSPITAL                   04   
112481         ROLLING HILLS PSYCHIATRIC HOSPITAL                   04   
112482            WESTERN MENTAL HEALTH INSTITUTE                   04   
112483            MIDDLE TN MENTAL HLTH INSTITUTE                   04   

                                 ST_ADR     CITY_NAME ZIP_CD  
112474             5908 LYONS VIEW PIKE     KNOXVILLE  37919  
11

In [34]:
import geopandas as gpd

# ── 1. Combine psychiatric hospitals and community mental health centers ───────
psych_hospitals = tn[
    (tn["PRVDR_CTGRY_CD"] == "01") &
    (tn["PRVDR_CTGRY_SBTYP_CD"].isin(["04", "07", "24", "25"]))
].copy()

community_mh = tn[tn["PRVDR_CTGRY_CD"] == "19"].copy()

df_mh = pd.concat([psych_hospitals, community_mh], ignore_index=True)

# ── 2. Drop closed facilities ─────────────────────────────────────────────────
df_mh = df_mh[~df_mh["FAC_NAME"].str.startswith("(CLOSED)", na=False)].copy()
print(f"Active mental health facilities in TN: {len(df_mh)}")

# ── 3. Geocode via Census batch geocoder ──────────────────────────────────────
df_mh = df_mh.reset_index(drop=True)
df_addr = pd.DataFrame({
    "id": df_mh.index.astype(str),
    "street": df_mh["ST_ADR"].fillna(""),
    "city": df_mh["CITY_NAME"].fillna(""),
    "state": df_mh["STATE_CD"].fillna(""),
    "zip": df_mh["ZIP_CD"].str[:5].fillna(""),
})

print(f"Geocoding {len(df_addr)} facilities...")
geo_results = census_batch_geocode(df_addr, "mh")
print(geo_results["match"].value_counts())

Active mental health facilities in TN: 39
Geocoding 39 facilities...
match
Match       38
No_Match     1
Name: count, dtype: int64


In [35]:
# ── 4. Parse coordinates ──────────────────────────────────────────────────────
matched = geo_results[geo_results["match"] == "Match"].copy()
matched[["lon", "lat"]] = matched["coords"].str.split(",", expand=True).astype(float)
matched["id"] = matched["id"].astype(int)

df_mh = df_mh.merge(matched[["id", "lat", "lon"]], left_index=True, right_on="id", how="left")
df_mh = df_mh.dropna(subset=["lat", "lon"])

# ── 5. Spatial join to tracts ─────────────────────────────────────────────────
gdf_mh = gpd.GeoDataFrame(
    df_mh,
    geometry=gpd.points_from_xy(df_mh["lon"], df_mh["lat"]),
    crs="EPSG:4326",
)

gdf_tracts = gdf_tracts.to_crs("EPSG:4326")

joined = gpd.sjoin(
    gdf_mh[["PRVDR_NUM", "geometry"]],
    gdf_tracts[["GEOID", "geometry"]],
    how="left",
    predicate="within",
)

mh_counts = (
    joined.groupby("GEOID")["PRVDR_NUM"]
    .count()
    .reset_index()
    .rename(columns={"PRVDR_NUM": "mh_facility_count"})
)

# ── 6. Merge and compute rate ─────────────────────────────────────────────────
gdf_tracts = gdf_tracts.merge(mh_counts, on="GEOID", how="left")
gdf_tracts["mh_facility_count"] = gdf_tracts["mh_facility_count"].fillna(0)

gdf_tracts["mh_facilities_per_10k"] = (
    gdf_tracts["mh_facility_count"] / gdf_tracts["TOTALPOP"] * 10_000
).where(gdf_tracts["TOTALPOP"] > 0)

print(gdf_tracts[["GEOID", "mh_facility_count", "mh_facilities_per_10k"]].describe())

# ── 7. Export ─────────────────────────────────────────────────────────────────
mh_export = gdf_tracts[["GEOID", "mh_facility_count", "mh_facilities_per_10k", "geometry"]].copy()
mh_export.to_file("/Users/maxbdixon/Desktop/SCHOOL/BRIC data/tn_mh_facilities_per_10k.gpkg", driver="GPKG")
print("Exported tn_mh_facilities_per_10k.gpkg")

       mh_facility_count  mh_facilities_per_10k
count        1471.000000            1471.000000
mean            0.024473               0.068605
std             0.163130               0.566537
min             0.000000               0.000000
25%             0.000000               0.000000
50%             0.000000               0.000000
75%             0.000000               0.000000
max             2.000000              13.262599
Exported tn_mh_facilities_per_10k.gpkg


In [36]:
import pandas as pd
import geopandas as gpd
import requests
import io

FPATH = "/Users/maxbdixon/Downloads/npi_raw_201912/npidata_pfile_20050523-20191219.csv"

COLS = [
    "NPI",
    "Entity Type Code",
    "Provider Business Practice Location Address State Name",
    "Provider First Line Business Practice Location Address",
    "Provider Business Practice Location Address City Name",
    "Provider Business Practice Location Address Postal Code",
    "Healthcare Provider Taxonomy Code_1",
    "Healthcare Provider Primary Taxonomy Switch_1",
]

# ── 1. Read in chunks, filtering to TN ───────────────────────────────────────
chunks = []
for chunk in pd.read_csv(FPATH, usecols=COLS, dtype=str, chunksize=100_000):
    chunk = chunk[
        (chunk["Entity Type Code"] == "1") &
        (chunk["Provider Business Practice Location Address State Name"] == "TN")
    ]
    chunks.append(chunk)

df = pd.concat(chunks, ignore_index=True)

# ── 2. Filter to mental health providers via NUCC taxonomy codes ──────────────
# 101   = Psychologist
# 1041  = Licensed Clinical Social Worker  
# 106   = Counselor
# 174M  = Marriage & Family Therapist
# 2084  = Psychiatrist (subset of 207)
# 3233  = Psychiatric/Mental Health NP
# 207P  = Psychiatry (broad, already in physician file)

MH_PREFIXES = ("101", "1041", "106", "174M", "2084", "3233", "207P")

is_primary = df["Healthcare Provider Primary Taxonomy Switch_1"] == "Y"
is_mh = df["Healthcare Provider Taxonomy Code_1"].str.startswith(MH_PREFIXES, na=False)

df_mh = df[is_primary & is_mh].copy()

print(f"TN mental health providers: {len(df_mh)}")
print(df_mh["Healthcare Provider Taxonomy Code_1"].value_counts().head(15))

TN mental health providers: 11146
Healthcare Provider Taxonomy Code_1
101YM0800X    3026
1041C0700X    1768
104100000X    1274
101YP2500X    1031
101Y00000X     978
207P00000X     893
2084P0800X     548
106H00000X     464
2084N0400X     297
106S00000X     265
101YA0400X     263
101YS0200X      58
207PP0204X      37
2084P0804X      36
2084N0402X      36
Name: count, dtype: int64


In [37]:
# ── 3. Geocode via Census batch geocoder ──────────────────────────────────────
df_mh = df_mh.reset_index(drop=True)
df_addr = pd.DataFrame({
    "id": df_mh.index.astype(str),
    "street": df_mh["Provider First Line Business Practice Location Address"].fillna(""),
    "city": df_mh["Provider Business Practice Location Address City Name"].fillna(""),
    "state": df_mh["Provider Business Practice Location Address State Name"].fillna(""),
    "zip": df_mh["Provider Business Practice Location Address Postal Code"].str[:5].fillna(""),
})

# run in batches of 9,500
results = []
n_batches = (len(df_addr) // 9_500) + 1

for i in range(n_batches):
    batch = df_addr.iloc[i * 9_500 : (i + 1) * 9_500]
    if len(batch) == 0:
        continue
    print(f"Geocoding batch {i+1}/{n_batches} ({len(batch)} records)...")
    result = census_batch_geocode(batch, i)
    results.append(result)

geo_results = pd.concat(results, ignore_index=True)
print(geo_results["match"].value_counts())

# ── 4. Parse coordinates ──────────────────────────────────────────────────────
matched = geo_results[geo_results["match"] == "Match"].copy()
matched[["lon", "lat"]] = matched["coords"].str.split(",", expand=True).astype(float)
matched["id"] = matched["id"].astype(int)

df_mh = df_mh.merge(matched[["id", "lat", "lon"]], left_index=True, right_on="id", how="left")
df_mh = df_mh.dropna(subset=["lat", "lon"])

# ── 5. Spatial join to tracts ─────────────────────────────────────────────────
gdf_mh = gpd.GeoDataFrame(
    df_mh,
    geometry=gpd.points_from_xy(df_mh["lon"], df_mh["lat"]),
    crs="EPSG:4326",
)

gdf_tracts = gdf_tracts.to_crs("EPSG:4326")

joined = gpd.sjoin(
    gdf_mh[["NPI", "geometry"]],
    gdf_tracts[["GEOID", "geometry"]],
    how="left",
    predicate="within",
)

mh_counts = (
    joined.groupby("GEOID")["NPI"]
    .count()
    .reset_index()
    .rename(columns={"NPI": "mh_provider_count"})
)

# ── 6. Merge and compute rate ─────────────────────────────────────────────────
gdf_tracts = gdf_tracts.merge(mh_counts, on="GEOID", how="left")
gdf_tracts["mh_provider_count"] = gdf_tracts["mh_provider_count"].fillna(0)

gdf_tracts["mh_providers_per_10k"] = (
    gdf_tracts["mh_provider_count"] / gdf_tracts["TOTALPOP"] * 10_000
).where(gdf_tracts["TOTALPOP"] > 0)

# ── 7. Winsorize ──────────────────────────────────────────────────────────────
p97 = gdf_tracts["mh_providers_per_10k"].quantile(0.97)
print(f"97th percentile: {p97:.1f}")
gdf_tracts["mh_providers_per_10k_w"] = gdf_tracts["mh_providers_per_10k"].clip(upper=p97)

print(gdf_tracts[["mh_provider_count", "mh_providers_per_10k", "mh_providers_per_10k_w"]].describe())

# ── 8. Export ─────────────────────────────────────────────────────────────────
mh_export = gdf_tracts[["GEOID", "mh_provider_count", "mh_providers_per_10k",
                         "mh_providers_per_10k_w", "geometry"]].copy()
mh_export.to_file("/Users/maxbdixon/Desktop/SCHOOL/BRIC data/tn_mh_providers_per_10k.gpkg", driver="GPKG")
print("Exported tn_mh_providers_per_10k.gpkg")

Geocoding batch 1/2 (9500 records)...
Geocoding batch 2/2 (1646 records)...
match
Match       10222
No_Match      851
Tie            73
Name: count, dtype: int64
97th percentile: 122.0
       mh_provider_count  mh_providers_per_10k  mh_providers_per_10k_w
count        1471.000000           1471.000000             1471.000000
mean            6.892590             18.912009               12.243065
std            27.312889             91.727468               26.490553
min             0.000000              0.000000                0.000000
25%             0.000000              0.000000                0.000000
50%             1.000000              2.268088                2.268088
75%             4.000000              9.261668                9.261668
max           747.000000           2122.159091              122.021568
Exported tn_mh_providers_per_10k.gpkg


In [38]:
# ── 1. Filter to hospitals in TN ──────────────────────────────────────────────
tn_hospitals = tn[
    (tn["PRVDR_CTGRY_CD"] == "01") &
    (tn["PRVDR_CTGRY_SBTYP_CD"] != "04")  # exclude pure psychiatric hospitals
].copy()

# drop closed facilities
tn_hospitals = tn_hospitals[~tn_hospitals["FAC_NAME"].str.startswith("(CLOSED)", na=False)]

print(f"TN hospitals: {len(tn_hospitals)}")
print(tn_hospitals["PRVDR_CTGRY_SBTYP_CD"].value_counts())
print(tn_hospitals[["FAC_NAME", "BED_CNT"]].head(10))

TN hospitals: 193
PRVDR_CTGRY_SBTYP_CD
01    136
11     19
02     12
05     11
20      8
06      3
Name: count, dtype: int64
                                       FAC_NAME BED_CNT
111473                  UNICOI COUNTY  HOSPITAL      48
111474  JACKSON-MADISON COUNTY GENERAL HOSPITAL     642
111475           SUMNER REGIONAL MEDICAL CENTER     155
111476                 EMERALD HODGSON HOSPITAL      42
111477           TRISTAR SKYLINE MEDICAL CENTER     395
111478                     UNITY MEDICAL CENTER      49
111479      HENDERSON COUNTY COMMUNITY HOSPITAL      45
111480                CUMBERLAND MEDICAL CENTER     189
111481                     WAYNE MEDICAL CENTER      80
111482                 BLOUNT MEMORIAL HOSPITAL     304


In [39]:
# ── 2. Geocode ────────────────────────────────────────────────────────────────
tn_hospitals = tn_hospitals.reset_index(drop=True)
df_addr = pd.DataFrame({
    "id": tn_hospitals.index.astype(str),
    "street": tn_hospitals["ST_ADR"].fillna(""),
    "city": tn_hospitals["CITY_NAME"].fillna(""),
    "state": tn_hospitals["STATE_CD"].fillna(""),
    "zip": tn_hospitals["ZIP_CD"].str[:5].fillna(""),
})

print(f"Geocoding {len(df_addr)} hospitals...")
geo_results_hosp = census_batch_geocode(df_addr, "hosp")
print(geo_results_hosp["match"].value_counts())

# ── 3. Parse coordinates ──────────────────────────────────────────────────────
matched = geo_results_hosp[geo_results_hosp["match"] == "Match"].copy()
matched[["lon", "lat"]] = matched["coords"].str.split(",", expand=True).astype(float)
matched["id"] = matched["id"].astype(int)

tn_hospitals = tn_hospitals.merge(
    matched[["id", "lat", "lon"]], left_index=True, right_on="id", how="left"
)
tn_hospitals = tn_hospitals.dropna(subset=["lat", "lon"])
tn_hospitals["BED_CNT"] = pd.to_numeric(tn_hospitals["BED_CNT"], errors="coerce").fillna(0)

# ── 4. Spatial join to tracts ─────────────────────────────────────────────────
gdf_hosp = gpd.GeoDataFrame(
    tn_hospitals,
    geometry=gpd.points_from_xy(tn_hospitals["lon"], tn_hospitals["lat"]),
    crs="EPSG:4326",
)

joined = gpd.sjoin(
    gdf_hosp[["PRVDR_NUM", "BED_CNT", "geometry"]],
    gdf_tracts[["GEOID", "geometry"]],
    how="left",
    predicate="within",
)

# sum beds per tract
bed_counts = (
    joined.groupby("GEOID")
    .agg(hospital_count=("PRVDR_NUM", "count"), bed_count=("BED_CNT", "sum"))
    .reset_index()
)

# ── 5. Merge and compute rate ─────────────────────────────────────────────────
gdf_tracts = gdf_tracts.merge(bed_counts, on="GEOID", how="left")
gdf_tracts["hospital_count"] = gdf_tracts["hospital_count"].fillna(0)
gdf_tracts["bed_count"] = gdf_tracts["bed_count"].fillna(0)

gdf_tracts["beds_per_10k"] = (
    gdf_tracts["bed_count"] / gdf_tracts["TOTALPOP"] * 10_000
).where(gdf_tracts["TOTALPOP"] > 0)

p97 = gdf_tracts["beds_per_10k"].quantile(0.97)
print(f"97th percentile: {p97:.1f}")
gdf_tracts["beds_per_10k_w"] = gdf_tracts["beds_per_10k"].clip(upper=p97)

print(gdf_tracts[["bed_count", "beds_per_10k", "beds_per_10k_w"]].describe())

# ── 6. Export ─────────────────────────────────────────────────────────────────
bed_export = gdf_tracts[["GEOID", "hospital_count", "bed_count",
                          "beds_per_10k", "beds_per_10k_w", "geometry"]].copy()
bed_export.to_file("/Users/maxbdixon/Desktop/SCHOOL/BRIC data/tn_beds_per_10k.gpkg", driver="GPKG")
print("Exported tn_beds_per_10k.gpkg")

Geocoding 193 hospitals...
match
Match       163
No_Match     28
Tie           2
Name: count, dtype: int64
97th percentile: 314.0
         bed_count  beds_per_10k  beds_per_10k_w
count  1471.000000   1471.000000     1471.000000
mean     17.783141     59.072440       16.313599
std     126.469533    693.530365       63.199961
min       0.000000      0.000000        0.000000
25%       0.000000      0.000000        0.000000
50%       0.000000      0.000000        0.000000
75%       0.000000      0.000000        0.000000
max    3095.000000  22135.278515      313.963584
Exported tn_beds_per_10k.gpkg


In [40]:
# look at S code breakdown
s_orgs = df_bmf_tn[
    (df_bmf_tn["NTEE_NCCS"].str.startswith("S", na=False)) &
    (df_bmf_tn["ORG_YEAR_FIRST"].astype(float) <= 2019) &
    (df_bmf_tn["ORG_YEAR_LAST"].astype(float) >= 2019)
]
print(s_orgs["NTEE_NCCS"].value_counts().head(20))

NTEE_NCCS
S80      558
S41      428
S20      361
S99       87
S30       75
S21       74
S82       67
S81       64
S03       56
S40       53
S22       44
S01       38
S31       31
S47       25
S12       20
S46       20
S02       18
S0341     15
S11       13
S50       13
Name: count, dtype: int64


In [41]:
import geopandas as gpd

gdf_precincts = gpd.read_file("/Users/maxbdixon/Downloads/tn_2018/tn_2018.shp")

print(gdf_precincts.shape)
print(gdf_precincts.columns.tolist())
print(gdf_precincts.head(3))

(1983, 11)
['STATEFP', 'COUNTYFP', 'NAME', 'VTD', 'G18GOVRLEE', 'G18GOVDDEA', 'G18GOVOOTH', 'G18USSRBLA', 'G18USSDBRE', 'G18USSOOTH', 'geometry']
  STATEFP COUNTYFP                 NAME   VTD  G18GOVRLEE  G18GOVDDEA  \
0      47      065           Red Bank 4   NaN           0           0   
1      47      065  North Chattanooga 1  3310         573         824   
2      47      065         Courthouse 2  3317          12          31   

   G18GOVOOTH  G18USSRBLA  G18USSDBRE  G18USSOOTH  \
0           0           0           0           0   
1          13         477         918          15   
2           0           7          35           0   

                                            geometry  
0  MULTIPOLYGON Z (((2177019.297 273500.535 0, 21...  
1  MULTIPOLYGON Z (((2182359.364 264138.408 0, 21...  
2  POLYGON Z ((2175248.891 259389.438 0, 2175269....  


In [42]:
import geopandas as gpd
import numpy as np

# ── 1. Compute total votes per precinct ───────────────────────────────────────
# use governor race as the primary turnout measure (higher participation than senate)
gdf_precincts["total_votes"] = (
    gdf_precincts["G18GOVRLEE"] +
    gdf_precincts["G18GOVDDEA"] +
    gdf_precincts["G18GOVOOTH"]
)

# drop zero-vote precincts (uncontested or data missing)
gdf_precincts = gdf_precincts[gdf_precincts["total_votes"] > 0].copy()
print(f"Precincts with votes: {len(gdf_precincts)}")

# ── 2. Reproject to match tracts ──────────────────────────────────────────────
gdf_precincts = gdf_precincts.to_crs(gdf_tracts.crs)

# ── 3. Area-weighted interpolation of votes to tracts ─────────────────────────
# since precincts and tracts don't align, we split by intersection
# and allocate votes proportionally to area overlap
gdf_tracts_proj = gdf_tracts[["GEOID", "geometry"]].copy()

intersected = gpd.overlay(gdf_precincts[["total_votes", "geometry"]],
                          gdf_tracts_proj, how="intersection")

# compute area of each intersection piece
intersected["piece_area"] = intersected.geometry.area

# compute original precinct area
precinct_areas = gdf_precincts[["total_votes", "geometry"]].copy()
precinct_areas["precinct_area"] = precinct_areas.geometry.area
precinct_areas = precinct_areas[["total_votes", "precinct_area"]].reset_index(drop=True)

# merge precinct area back to get proportion
intersected = intersected.reset_index(drop=True)
intersected["precinct_area"] = intersected.groupby(
    intersected.index)["piece_area"].transform("sum")

# simpler: recompute from original precinct areas via spatial join
gdf_precincts["precinct_area"] = gdf_precincts.geometry.area
gdf_precincts = gdf_precincts.reset_index(drop=True)

intersected = gpd.overlay(
    gdf_precincts[["total_votes", "precinct_area", "geometry"]],
    gdf_tracts_proj,
    how="intersection"
)

intersected["piece_area"] = intersected.geometry.area
intersected["vote_share"] = intersected["piece_area"] / intersected["precinct_area"]
intersected["votes_allocated"] = intersected["total_votes"] * intersected["vote_share"]

# ── 4. Sum allocated votes to tracts ─────────────────────────────────────────
tract_votes = (
    intersected.groupby("GEOID")["votes_allocated"]
    .sum()
    .reset_index()
    .rename(columns={"votes_allocated": "total_votes_allocated"})
)

# ── 5. Merge onto tracts and compute turnout ──────────────────────────────────
# assumes your ACS data has voting age population as VAP or similar column
# check what your column is called
print([c for c in gdf_tracts.columns if "age" in c.lower() or "vap" in c.lower() or "vote" in c.lower()])

Precincts with votes: 1971


<positron-console-cell-42>:28: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

<positron-console-cell-42>:32: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

<positron-console-cell-42>:41: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

<positron-console-cell-42>:50: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.



['QAGEDEP', 'MEDAGE']


In [43]:
import pandas as pd

df_streets = pd.read_csv(
    "/Users/maxbdixon/Downloads/usa-tracts-street_network-stats.csv",
    dtype={"geoid": str}
)

df_streets["geoid"] = df_streets["geoid"].str.zfill(11)
df_streets_tn = df_streets[df_streets["geoid"].str.startswith("47")].copy()

print(f"TN tracts: {len(df_streets_tn)}")
print(df_streets_tn[["geoid", "intersect_density_km", "street_density_km", "node_density_km"]].describe())

TN tracts: 1497
       intersect_density_km  street_density_km  node_density_km
count           1497.000000        1497.000000      1497.000000
mean              16.417050        4460.072497        20.961616
std               17.641182        3325.440177        20.631262
min                0.137008         112.934392         0.158086
25%                2.254162        1554.821410         3.412017
50%                9.188333        3425.380976        13.761763
75%               25.798549        6850.917715        33.861034
max               92.520846       18162.750971        98.643549


In [44]:
# merge onto tracts
gdf_tracts = gdf_tracts.merge(
    df_streets_tn[["geoid", "intersect_density_km", "street_density_km"]].rename(columns={"geoid": "GEOID"}),
    on="GEOID",
    how="left"
)

print(f"Tracts missing street data: {gdf_tracts['intersect_density_km'].isna().sum()}")

# export
street_export = gdf_tracts[["GEOID", "intersect_density_km", "street_density_km", "geometry"]].copy()
street_export.to_file("/Users/maxbdixon/Desktop/SCHOOL/BRIC data/tn_street_network.gpkg", driver="GPKG")
print("Exported tn_street_network.gpkg")

Tracts missing street data: 0
Exported tn_street_network.gpkg


In [52]:
import geopandas as gpd

# ── 1. Load rail network ──────────────────────────────────────────────────────
gdf_rail = gpd.read_file(
    "/Users/maxbdixon/Downloads/NTAD_North_American_Rail_Network_Lines_6760165070091449209/North_American_Rail_Network_Lines.shp"
)

print(gdf_rail.shape)
print(gdf_rail.columns.tolist())
print(gdf_rail.head(3))

(302734, 33)
['FRAARCID', 'FRFRANODE', 'TOFRANODE', 'STFIPS', 'CNTYFIPS', 'STCNTYFIPS', 'STATEAB', 'COUNTRY', 'FRADISTRCT', 'RROWNER1', 'RROWNER2', 'RROWNER3', 'TRKRGHTS1', 'TRKRGHTS2', 'TRKRGHTS3', 'TRKRGHTS4', 'TRKRGHTS5', 'TRKRGHTS6', 'TRKRGHTS7', 'TRKRGHTS8', 'TRKRGHTS9', 'DIVISION', 'SUBDIV', 'BRANCH', 'YARDNAME', 'PASSNGR', 'STRACNET', 'TRACKS', 'NET', 'MILES', 'KM', 'TIMEZONE', 'geometry']
   FRAARCID  FRFRANODE  TOFRANODE STFIPS CNTYFIPS STCNTYFIPS STATEAB COUNTRY  \
0    300000     348741     348746     38      015      38015      ND      US   
1    300001     338567     338686     30      087      30087      MT      US   
2    300002     330112     330117     16      031      16031      ID      US   

   FRADISTRCT RROWNER1  ...      BRANCH YARDNAME PASSNGR STRACNET TRACKS NET  \
0           8     DMVW  ...       XLINE      NaN     NaN      NaN      1   M   
1           8     BNSF  ...         NaN      NaN     NaN      NaN      0   O   
2           8     EIRR  ...  TWIN FALLS

In [65]:
import pandas as pd

df_lead = pd.read_csv(
    "/Users/maxbdixon/Downloads/LEAD Tool Data Census Tracts (Mar 17, 2026 10_52am).csv",
    dtype={"Geography ID": str}
)

print(df_lead.shape)
print(df_lead["Geography ID"].head(15))
print(df_lead["Energy Burden (% income)"].describe())

(1701, 26)
0     47001020100
1     47001020201
2     47001020202
3     47001020300
4     47001020400
5     47001020500
6     47001020600
7     47001020700
8     47001020800
9     47001020901
10    47001020902
11    47001021001
12    47001021002
13    47001021100
14    47001021201
Name: Geography ID, dtype: str
count     1701
unique      11
top          3
freq       670
Name: Energy Burden (% income), dtype: object


In [66]:
print(df_lead["Energy Burden (% income)"].value_counts().head(10))

Energy Burden (% income)
3    670
2    527
4    259
1    116
5     61
6     24
-     23
7     12
8      4
9      4
Name: count, dtype: int64


In [70]:
# invert so higher = more resilient (lower burden = more efficient)
gdf_tracts["energy_burden_inv"] = 1 / gdf_tracts["energy_burden_pct"]
gdf_tracts.loc[gdf_tracts["energy_burden_pct"] == 0, "energy_burden_inv"] = None

print(gdf_tracts["energy_burden_inv"].describe())

energy_export = gdf_tracts[["GEOID", "energy_burden_pct", "energy_burden_inv", "geometry"]].copy()
energy_export.to_file("/Users/maxbdixon/Desktop/SCHOOL/BRIC data/tn_energy_burden.gpkg", driver="GPKG")
print("Exported tn_energy_burden.gpkg")


count    1268.000000
mean        0.399212
std         0.184543
min         0.111111
25%         0.333333
50%         0.333333
75%         0.500000
max         1.000000
Name: energy_burden_inv, dtype: float64
Exported tn_energy_burden.gpkg


In [71]:
import geopandas as gpd
import pandas as pd
import glob
import os

# ── 1. Load all tn_ gpkg files ────────────────────────────────────────────────
gpkg_dir = "/Users/maxbdixon/Desktop/SCHOOL/BRIC data"
gpkg_files = glob.glob(os.path.join(gpkg_dir, "tn_*.gpkg"))

print(f"Found {len(gpkg_files)} files:")
for f in sorted(gpkg_files):
    print(f"  {os.path.basename(f)}")

Found 7 files:
  tn_beds_per_10k.gpkg
  tn_energy_burden.gpkg
  tn_mh_providers_per_10k.gpkg
  tn_orgs_per_10k.gpkg
  tn_physicians_per_10k.gpkg
  tn_schools_per_10k.gpkg
  tn_street_network.gpkg


In [72]:
import geopandas as gpd
import pandas as pd
import glob
import os
import numpy as np

gpkg_dir = "/Users/maxbdixon/Desktop/SCHOOL/BRIC data"
gpkg_files = sorted(glob.glob(os.path.join(gpkg_dir, "tn_*.gpkg")))

results = []

for fpath in gpkg_files:
    name = os.path.basename(fpath).replace(".gpkg", "")
    gdf = gpd.read_file(fpath)
    
    # get numeric columns only, exclude GEOID and geometry
    num_cols = gdf.select_dtypes(include=[np.number]).columns.tolist()
    
    for col in num_cols:
        s = gdf[col].dropna()
        n_total = len(gdf)
        n_nonzero = (s > 0).sum()
        pct_zero = (s == 0).sum() / n_total * 100
        pct_missing = gdf[col].isna().sum() / n_total * 100
        cv = s.std() / s.mean() if s.mean() > 0 else np.nan  # coefficient of variation
        
        results.append({
            "file": name,
            "column": col,
            "n_nonzero": n_nonzero,
            "pct_zero": round(pct_zero, 1),
            "pct_missing": round(pct_missing, 1),
            "mean": round(s.mean(), 3),
            "median": round(s.median(), 3),
            "std": round(s.std(), 3),
            "cv": round(cv, 3) if not np.isnan(cv) else np.nan,
            "min": round(s.min(), 3),
            "max": round(s.max(), 3),
        })

df_diag = pd.DataFrame(results)

# focus on the key rate/density columns, exclude raw counts and winsorized duplicates
exclude = ["_count", "_w", "area_sq_mi"]
mask = ~df_diag["column"].str.contains("|".join(exclude), na=False)
df_diag = df_diag[mask]

pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
print(df_diag.to_string(index=False))

                   file                 column  n_nonzero  pct_zero  pct_missing     mean   median      std     cv     min       max
        tn_beds_per_10k           beds_per_10k        115      92.2          0.0   59.072    0.000  693.530 11.740   0.000 22135.279
       tn_energy_burden      energy_burden_pct       1268       0.0         13.8    2.933    3.000    1.128  0.384   1.000     9.000
       tn_energy_burden      energy_burden_inv       1268       0.0         13.8    0.399    0.333    0.185  0.462   0.111     1.000
tn_mh_providers_per_10k   mh_providers_per_10k        882      40.0          0.0   18.912    2.268   91.727  4.850   0.000  2122.159
        tn_orgs_per_10k     civic_orgs_per_10k        831      43.5          0.0    3.580    1.664    7.895  2.205   0.000   152.259
        tn_orgs_per_10k religious_orgs_per_10k       1314      10.7          0.0    8.993    6.499   10.050  1.117   0.000   124.167
  tn_physicians_per_10k     physicians_per_10k        728      50.5  

In [73]:
import geopandas as gpd
import numpy as np

TN_CRS = "EPSG:3968"

# ── 1. Get geocoded hospitals from earlier ────────────────────────────────────
gdf_hosp = gpd.GeoDataFrame(
    tn_hospitals.dropna(subset=["lat", "lon"]),
    geometry=gpd.points_from_xy(tn_hospitals.dropna(subset=["lat", "lon"])["lon"],
                                tn_hospitals.dropna(subset=["lat", "lon"])["lat"]),
    crs="EPSG:4326"
).to_crs(TN_CRS)

# ── 2. Compute tract centroids ────────────────────────────────────────────────
tracts_proj = gdf_tracts[["GEOID", "geometry"]].to_crs(TN_CRS).copy()
tracts_proj["centroid"] = tracts_proj.geometry.centroid

gdf_centroids = gpd.GeoDataFrame(
    tracts_proj[["GEOID"]],
    geometry=tracts_proj["centroid"],
    crs=TN_CRS
)

# ── 3. Distance to nearest hospital for each tract centroid ───────────────────
from shapely.ops import nearest_points

hosp_union = gdf_hosp.geometry.unary_union

def dist_to_nearest(point):
    nearest = nearest_points(point, hosp_union)[1]
    return point.distance(nearest)

gdf_centroids["dist_to_hosp_m"] = gdf_centroids.geometry.apply(dist_to_nearest)
gdf_centroids["dist_to_hosp_mi"] = gdf_centroids["dist_to_hosp_m"] / 1609.344

print(gdf_centroids["dist_to_hosp_mi"].describe())

# ── 4. Merge onto tracts ──────────────────────────────────────────────────────
gdf_tracts = gdf_tracts.merge(
    gdf_centroids[["GEOID", "dist_to_hosp_mi"]],
    on="GEOID",
    how="left"
)

# ── 5. Invert — closer is more resilient ─────────────────────────────────────
gdf_tracts["hosp_proximity"] = 1 / (gdf_tracts["dist_to_hosp_mi"] + 1)

print(gdf_tracts[["dist_to_hosp_mi", "hosp_proximity"]].describe())

# ── 6. Export ─────────────────────────────────────────────────────────────────
hosp_export = gdf_tracts[["GEOID", "dist_to_hosp_mi", "hosp_proximity", "geometry"]].copy()
hosp_export.to_file("/Users/maxbdixon/Desktop/SCHOOL/BRIC data/tn_hosp_proximity.gpkg", driver="GPKG")
print("Exported tn_hosp_proximity.gpkg")

<positron-console-cell-73>:27: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.


count    1471.000000
mean        6.041707
std         4.751992
min         0.062042
25%         2.378484
50%         4.796791
75%         8.655460
max        30.418786
Name: dist_to_hosp_mi, dtype: float64
       dist_to_hosp_mi  hosp_proximity
count      1471.000000     1471.000000
mean          6.041707        0.222580
std           4.751992        0.157298
min           0.062042        0.031828
25%           2.378484        0.103568
50%           4.796791        0.172509
75%           8.655460        0.295991
max          30.418786        0.941583
Exported tn_hosp_proximity.gpkg
